In [1]:
import pandas as pd
import numpy as np

In [3]:
# Load dataset
daily_df = pd.read_csv('dailyActivity_merged.csv')
sleep_df = pd.read_csv('sleepDay_merged.csv')

daily_df.head()

,Id,ActivityDate,TotalSteps,TotalDistance,TrackerDistance,LoggedActivitiesDistance,VeryActiveDistance,ModeratelyActiveDistance,LightActiveDistance,SedentaryActiveDistance,VeryActiveMinutes,FairlyActiveMinutes,LightlyActiveMinutes,SedentaryMinutes,Calories,NonWearFlag,TotalActiveMinutes
0,1503960366,3/25/2016,11004,7.11,7.11,0.0,2.57,0.46,4.07,0.0,33,12,205,804,1819,Valid,250
1,1503960366,3/26/2016,17609,11.55,11.55,0.0,6.92,0.73,3.91,0.0,89,17,274,588,2154,Valid,380
2,1503960366,3/27/2016,12736,8.53,8.53,0.0,4.66,0.16,3.71,0.0,56,5,268,605,1944,Valid,329
3,1503960366,3/28/2016,13231,8.93,8.93,0.0,3.19,0.79,4.95,0.0,39,20,224,1080,1932,Valid,283
4,1503960366,3/29/2016,12041,7.85,7.85,0.0,2.16,1.09,4.61,0.0,28,28,243,763,1886,Valid,299


In [4]:
sleep_df.head()

,Id,SleepDay,TotalSleepRecords,TotalMinutesAsleep,TotalTimeInBed,Date
0,1503960366,04-12-2016 00:00,1,327,346,2016-04-12
1,1503960366,4/13/2016 12:00:00 AM,2,384,407,2016-04-13
2,1503960366,4/15/2016 12:00:00 AM,1,412,442,2016-04-15
3,1503960366,4/16/2016 12:00:00 AM,2,340,367,2016-04-16
4,1503960366,4/17/2016 12:00:00 AM,1,700,712,2016-04-17


In [7]:
# Format dates & duplicates
daily_df['ActivityDate'] = pd.to_datetime(daily_df['ActivityDate'], format='mixed')
sleep_df.drop_duplicates(inplace=True)
sleep_df['SleepDay'] = pd.to_datetime(sleep_df['SleepDay'], format='mixed')

In [8]:
# Feature Engineering
daily_df['TotalActiveMinutes'] = (
    daily_df['VeryActiveMinutes'] +
    daily_df['FairlyActiveMinutes'] +
    daily_df['LightlyActiveMinutes']
)
daily_df['DayOfWeek'] = daily_df['ActivityDate'].dt.day_name()
sleep_df['BedWaitTimeMinutes'] = sleep_df['TotalTimeInBed'] - sleep_df['TotalMinutesAsleep']

In [11]:
# Baseline descriptive Statistics
summary_stats = daily_df[['TotalSteps', 'TotalDistance', 'SedentaryMinutes', 'TotalActiveMinutes', 'Calories']].describe().round(2)
print("=== Daily Activity Descriptive Metrics ===")
print(summary_stats)

=== Daily Activity Descriptive Metrics ===
       TotalSteps  TotalDistance  SedentaryMinutes  TotalActiveMinutes  \
count      457.00         457.00            457.00              457.00   
mean      6546.56           4.66            995.28              199.76   
std       5398.49           4.08            337.02              140.32   
min          0.00           0.00             32.00                0.00   
25%       1988.00           1.41            728.00               74.00   
50%       5986.00           4.09           1057.00              221.00   
75%      10198.00           7.16           1285.00              301.00   
max      28497.00          27.53           1440.00              720.00   

       Calories  
count    457.00  
mean    2189.45  
std      815.48  
min        0.00  
25%     1776.00  
50%     2062.00  
75%     2667.00  
max     4562.00  


In [12]:
# Behavioural Tiering
user_groups = daily_df.groupby('Id').agg({
    'TotalSteps': 'mean',
    'SedentaryMinutes': 'mean',
    'Calories': 'mean'
}).reset_index()

user_groups['UserTier'] = pd.cut(
    user_groups['TotalSteps'],
    bins=[-np.inf, 4999, 7499, 9999, np.inf],
    labels=['Sedentary (<5k)', 'Low Active (5k-7.5k)', 'Fairly Active (7.5k-10k)', 'Highly Active (>10k)']
)
print("\n=== User Segmentation Breakdown ===")
print(user_groups['UserTier'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')


=== User Segmentation Breakdown ===
UserTier
Sedentary (<5k)             40.0%
Fairly Active (7.5k-10k)    25.7%
Low Active (5k-7.5k)        17.1%
Highly Active (>10k)        17.1%
Name: proportion, dtype: object
